In [ ]:
"""Daily Timeframe
EMA rebound Scanner - Daily Level
Strategy: Signal triggered when the candle's lower wick touches the EMA but the close remains above it
"""

import os
from datetime import datetime, timedelta
from polygon import RESTClient
import pandas as pd

# ========== Enter your API Key here ==========
POLYGON_API_KEY = "Your API Key"
# ==============================================

# ========== Modify your watchlist here ==========
watchlist=["AAPL","ABBV","ABNB","ABT","ACGL","ACN","ADBE","ADI","ADM","ADP","ADSK","AEE","AEP","AES","AFL","AIG",
    "AIZ","AJG","AKAM","ALB","ALGN","ALL","ALLE","AMAT","AMCR","AMD","AME","AMGN","AMP","AMT","AMZN","ANET",
    "ANSS","AON","AOS","APA","APD","APH","APTV","ARE","ATO","AVB","AVGO","AVY","AWK","AXON","AXP","AZO","BA",
    "BAC","BALL","BAX","BBWI","BBY","BDX","BEN","BF-B","BIIB","BK","BKNG","BKR","BLK","BMY","BR","BRK-B","BRO",
    "BSX","BWA","BX","BXP","C","CAG","CAH","CARR","CAT","CB","CBOE","CBRE","CCI","CCL","CDNS","CDW","CE","CEG",
    "CF","CFG","CHD","CHRW","CHTR","CI","CINF","CL","CLX","CMA","CMCSA","CME","CMG","CMI","CMS","CNC","CNP","COF",
    "COO","COP","COST","CPB","CPRT","CPT","CRL","CRM","CRWD","CSCO","CSGP","CSX","CTAS","CTLT","CTRA","CTSH","CTVA",
    "CVS","CVX","D","DAL","DD","DE","DFS","DG","DGX","DHI","DHR","DIS","DLR","DLTR","DOV","DOW","DPZ","DRI","DTE",
    "DUK","DVA","DVN","DXCM","EA","EBAY","ECL","ED","EFX","EIX","EL","ELV","EMN","EMR","ENPH","EOG","EPAM","EQIX","EQR",
    "EQT","ES","ESS","ETN","ETR","EVRG","EW","EXC","EXPD","EXPE","EXR","F","FANG","FAST","FCX","FDS","FDX","FE","FFIV",
    "FICO","FIS","FISV","FITB","FLT","FMC","FOXA","FRT","FSLR","GD","GE","GEHC","GEN","GENE","GILD","GIS","GL","GLW","GM",
    "GOOGL","GPC","GPN","GRMN","GS","GWW","HAL","HAS","HBAN","HCA","HD","HES","HIG","HII","HLT","HOLX","HON","HPE","HPQ",
    "HRL","HSIC","HST","HSY","HUBB","HUM","HWM","IBM","ICE","IDXX","IEX","IFF","ILMN","INCY","INTC","INTU","IP","IPG","IQV",
    "IR","IRM","ISRG","IT","ITW","IVZ","J","JBHT","JCI","JKHY","JNJ","JPM","K","KDP","KEY","KEYS","KHC","KIM","KLAC","KMB",
    "KMI","KMX","KO","KR","L","LDOS","LEN","LH","LHX","LIN","LKQ","LLY","LMT","LNT","LOW","LRCX","LULU","LUV","LVS","LW","LYB",
    "LYV","MA","MAA","MAR","MAS","MCD","MCHP","MCK","MCO","MDLZ","MDT","MET","META","MGM","MHK","MKC","MKTX","MLM","MMC","MMM",
    "MNST","MO","MOS","MPC","MPWR","MRK","MRNA","MRO","MS","MSCI","MSFT","MSI","MTB","MTD","MU","NCLH","NDAQ","NEE","NEM","NFLX",
    "NI","NKE","NOC","NOW","NRG","NSC","NTAP","NTRS","NUE","NVDA","NVR","NWL","NWSA","NXPI","O","ODFL","OKE","OMC","ON","ORCL",
    "ORLY","OTIS","OXY","PANW","PARA","PAYC","PAYX","PCAR","PCG","PEAK","PEG","PEP","PFE","PFG","PG","PGR","PH","PHM","PKG","PLD",
    "PM","PNC","PNR","PNW","PODD","POOL","PPG","PPL","PRU","PSA","PSX","PTC","PWR","PYPL","QCOM","RCL","REG","REGN","RF","RHI","RJF",
    "RL","RMD","ROK","ROL","ROP","ROST","RSG","RTX","SBUX","SCHW","SEDG","SEE","SHW","SJM","SLB","SMCI","SNA","SNPS","SO","SPG","SPGI",
    "SRE","STE","STT","STX","STZ","SWK","SWKS","SYF","SYK","SYY","T","TAP","TDG","TDY","TECH","TEL","TER","TFC","TFX","TGT","TJX","TMO",
    "TMUS","TPR","TRGP","TRMB","TROW","TRV","TSCO","TSLA","TSN","TT","TTWO","TXN","TXT","TYL","UAL","UDR","UHS","ULTA","UNH","UNP","UPS",
    "URI","USB","V","VFC","VICI","VLO","VMC","VRSK","VRSN","VRTX","VTR","VTRS","WAB","WAT","WBA","WBD","WDC","WEC","WELL","WFC","WHR","WM",
    "WMB","WMT","WRB","WY","WYNN","USAR","MP","LAC","XEL","XOM","XYL","YUM","ZBH","ZBRA","ZTS","BJ"]
# ================================================

class EMAScanner:
    def __init__(self, api_key: str = None):
        key = api_key or POLYGON_API_KEY
        if key == "YOUR_API_KEY_HERE":
            raise ValueError("Please enter your POLYGON_API_KEY at the top of the script")
        self.client = RESTClient(key)
    
    def get_stock_bars(self, symbol: str) -> pd.DataFrame:
        """Fetch daily bar data"""
        end_date = datetime.now()
        start_date = end_date - timedelta(days=5000)
        
        try:
            bars = self.client.get_aggs(
                ticker=symbol,
                multiplier=1,
                timespan="day",
                from_=start_date.strftime("%Y-%m-%d"),
                to=end_date.strftime("%Y-%m-%d"),
                limit=50000
            )
            
            if not bars:
                return pd.DataFrame()
            
            df = pd.DataFrame([{
                'timestamp': bar.timestamp,
                'open': bar.open,
                'high': bar.high,
                'low': bar.low,
                'close': bar.close,
                'volume': bar.volume
            } for bar in bars])
            
            df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
            df = df.sort_values('date').reset_index(drop=True)
            
            return df
            
        except Exception as e:
            print(f"Error fetching data for {symbol}: {e}")
            return pd.DataFrame()
    
    def calculate_ema(self, df: pd.DataFrame, period: int) -> pd.DataFrame:
        if df.empty:
            return df
        df = df.copy()
        df[f'ema_{period}'] = df['close'].ewm(span=period, adjust=False).mean()
        return df
    
    def check_ema_bounce_signal(self, df: pd.DataFrame, ema_period: int) -> bool:
        if df.empty or f'ema_{ema_period}' not in df.columns:
            return False
        latest = df.iloc[-1]
        ema_value = latest[f'ema_{ema_period}']
        return latest['low'] <= ema_value and latest['close'] > ema_value
    
    def scan_single_stock(self, symbol: str) -> dict:
        df = self.get_stock_bars(symbol)
        if df.empty:
            return None
        
        df = self.calculate_ema(df, 100)
        df = self.calculate_ema(df, 200)
        
        signal_100 = self.check_ema_bounce_signal(df, 100)
        signal_200 = self.check_ema_bounce_signal(df, 200)
        
        if signal_100 or signal_200:
            return {'ticker': symbol, '100ema': signal_100, '200ema': signal_200}
        return None
    
    def scan_multiple_stocks(self, symbols: list) -> pd.DataFrame:
        results = []
        total = len(symbols)
        
        print(f"\nScanning {total} stocks (daily timeframe)...")
        print("-" * 60)
        
        for i, symbol in enumerate(symbols, 1):
            print(f"Scanning [{i}/{total}]: {symbol}", end="\r")
            signal = self.scan_single_stock(symbol)
            if signal:
                results.append(signal)
        
        print(f"\nScan complete! Found {len(results)} signal(s)")
        
        if results:
            return pd.DataFrame(results)
        return pd.DataFrame(columns=['ticker', '100ema', '200ema'])


def format_signals(df: pd.DataFrame) -> pd.DataFrame:
    """Format signal display: True shows checkmark, False shows blank"""
    if df.empty:
        return df
    
    df = df.copy()
    df['100ema'] = df['100ema'].apply(lambda x: '✅' if x else '')
    df['200ema'] = df['200ema'].apply(lambda x: '✅' if x else '')
    return df



# ========== Create save directory ==========
save_folder = "ema_data"
if not os.path.exists(save_folder):
    os.makedirs(save_folder)
# ============================================

# Run daily scan
scanner = EMAScanner()
daily_df = scanner.scan_multiple_stocks(watchlist)

today = datetime.now().strftime("%Y-%m-%d")
csv_filename = f"{today}_daily_ema.csv"
csv_path = os.path.join(save_folder, csv_filename)

print("\n📋 Daily Signals:")
if not daily_df.empty:
    display_df = format_signals(daily_df)
    display_df.to_csv(csv_path, index=False)
    print(display_df.to_string(index=False))
else:
    print("  No signals")
    daily_df.to_csv(csv_path, index=False)

print(f"\nResults exported: {csv_path}")

In [ ]:
"""周级别
EMA 触碰策略扫描器 - 周线级别
策略逻辑：当股价下影线触碰到 EMA，但收盘价在 EMA 之上时，发出信号
"""

import os
from datetime import datetime, timedelta
from polygon import RESTClient
import pandas as pd

# ========== 在这里填写你的 API Key ==========
POLYGON_API_KEY = "Your API Key"
# ============================================

# ========== 在这里修改你的观察列表 ==========
watchlist=["AAPL","ABBV","ABNB","ABT","ACGL","ACN","ADBE","ADI","ADM","ADP","ADSK","AEE","AEP","AES","AFL","AIG",
    "AIZ","AJG","AKAM","ALB","ALGN","ALL","ALLE","AMAT","AMCR","AMD","AME","AMGN","AMP","AMT","AMZN","ANET",
    "ANSS","AON","AOS","APA","APD","APH","APTV","ARE","ATO","AVB","AVGO","AVY","AWK","AXON","AXP","AZO","BA",
    "BAC","BALL","BAX","BBWI","BBY","BDX","BEN","BF-B","BIIB","BK","BKNG","BKR","BLK","BMY","BR","BRK-B","BRO",
    "BSX","BWA","BX","BXP","C","CAG","CAH","CARR","CAT","CB","CBOE","CBRE","CCI","CCL","CDNS","CDW","CE","CEG",
    "CF","CFG","CHD","CHRW","CHTR","CI","CINF","CL","CLX","CMA","CMCSA","CME","CMG","CMI","CMS","CNC","CNP","COF",
    "COO","COP","COST","CPB","CPRT","CPT","CRL","CRM","CRWD","CSCO","CSGP","CSX","CTAS","CTLT","CTRA","CTSH","CTVA",
    "CVS","CVX","D","DAL","DD","DE","DFS","DG","DGX","DHI","DHR","DIS","DLR","DLTR","DOV","DOW","DPZ","DRI","DTE",
    "DUK","DVA","DVN","DXCM","EA","EBAY","ECL","ED","EFX","EIX","EL","ELV","EMN","EMR","ENPH","EOG","EPAM","EQIX","EQR",
    "EQT","ES","ESS","ETN","ETR","EVRG","EW","EXC","EXPD","EXPE","EXR","F","FANG","FAST","FCX","FDS","FDX","FE","FFIV",
    "FICO","FIS","FISV","FITB","FLT","FMC","FOXA","FRT","FSLR","GD","GE","GEHC","GEN","GENE","GILD","GIS","GL","GLW","GM",
    "GOOGL","GPC","GPN","GRMN","GS","GWW","HAL","HAS","HBAN","HCA","HD","HES","HIG","HII","HLT","HOLX","HON","HPE","HPQ",
    "HRL","HSIC","HST","HSY","HUBB","HUM","HWM","IBM","ICE","IDXX","IEX","IFF","ILMN","INCY","INTC","INTU","IP","IPG","IQV",
    "IR","IRM","ISRG","IT","ITW","IVZ","J","JBHT","JCI","JKHY","JNJ","JPM","K","KDP","KEY","KEYS","KHC","KIM","KLAC","KMB",
    "KMI","KMX","KO","KR","L","LDOS","LEN","LH","LHX","LIN","LKQ","LLY","LMT","LNT","LOW","LRCX","LULU","LUV","LVS","LW","LYB",
    "LYV","MA","MAA","MAR","MAS","MCD","MCHP","MCK","MCO","MDLZ","MDT","MET","META","MGM","MHK","MKC","MKTX","MLM","MMC","MMM",
    "MNST","MO","MOS","MPC","MPWR","MRK","MRNA","MRO","MS","MSCI","MSFT","MSI","MTB","MTD","MU","NCLH","NDAQ","NEE","NEM","NFLX",
    "NI","NKE","NOC","NOW","NRG","NSC","NTAP","NTRS","NUE","NVDA","NVR","NWL","NWSA","NXPI","O","ODFL","OKE","OMC","ON","ORCL",
    "ORLY","OTIS","OXY","PANW","PARA","PAYC","PAYX","PCAR","PCG","PEAK","PEG","PEP","PFE","PFG","PG","PGR","PH","PHM","PKG","PLD",
    "PM","PNC","PNR","PNW","PODD","POOL","PPG","PPL","PRU","PSA","PSX","PTC","PWR","PYPL","QCOM","RCL","REG","REGN","RF","RHI","RJF",
    "RL","RMD","ROK","ROL","ROP","ROST","RSG","RTX","SBUX","SCHW","SEDG","SEE","SHW","SJM","SLB","SMCI","SNA","SNPS","SO","SPG","SPGI",
    "SRE","STE","STT","STX","STZ","SWK","SWKS","SYF","SYK","SYY","T","TAP","TDG","TDY","TECH","TEL","TER","TFC","TFX","TGT","TJX","TMO",
    "TMUS","TPR","TRGP","TRMB","TROW","TRV","TSCO","TSLA","TSN","TT","TTWO","TXN","TXT","TYL","UAL","UDR","UHS","ULTA","UNH","UNP","UPS",
    "URI","USB","V","VFC","VICI","VLO","VMC","VRSK","VRSN","VRTX","VTR","VTRS","WAB","WAT","WBA","WBD","WDC","WEC","WELL","WFC","WHR","WM",
    "WMB","WMT","WRB","WY","WYNN","USAR","MP","LAC","XEL","XOM","XYL","YUM","ZBH","ZBRA","ZTS","BJ","BK"]
# ============================================

class EMAScanner:
    def __init__(self, api_key: str = None):
        key = api_key or POLYGON_API_KEY
        if key == "YOUR_API_KEY_HERE":
            raise ValueError("请在代码顶部填写你的 POLYGON_API_KEY")
        self.client = RESTClient(key)
    
    def get_stock_bars(self, symbol: str) -> pd.DataFrame:
        """获取日线数据并转换为周线"""
        end_date = datetime.now()
        start_date = end_date - timedelta(days=5000)
        
        try:
            bars = self.client.get_aggs(
                ticker=symbol,
                multiplier=1,
                timespan="day",
                from_=start_date.strftime("%Y-%m-%d"),
                to=end_date.strftime("%Y-%m-%d"),
                limit=50000
            )
            
            if not bars:
                return pd.DataFrame()
            
            df = pd.DataFrame([{
                'timestamp': bar.timestamp,
                'open': bar.open,
                'high': bar.high,
                'low': bar.low,
                'close': bar.close,
                'volume': bar.volume
            } for bar in bars])
            
            df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
            df = df.sort_values('date').reset_index(drop=True)
            
            # 转换为周线
            df = self._convert_to_weekly(df)
            
            return df
            
        except Exception as e:
            print(f"获取 {symbol} 数据时出错: {e}")
            return pd.DataFrame()
    
    def _convert_to_weekly(self, df: pd.DataFrame) -> pd.DataFrame:
        """将日线数据转换为周线数据"""
        if df.empty:
            return df
        
        df = df.copy()
        df['weekday'] = df['date'].dt.dayofweek
        df['week_start'] = df['date'] - pd.to_timedelta(df['weekday'], unit='D')
        
        weekly = df.groupby('week_start').agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum',
            'date': 'last'
        }).reset_index()
        
        weekly = weekly.rename(columns={'week_start': 'week_date'})
        weekly = weekly.sort_values('week_date').reset_index(drop=True)
        
        return weekly
    
    def calculate_ema(self, df: pd.DataFrame, period: int) -> pd.DataFrame:
        if df.empty:
            return df
        df = df.copy()
        df[f'ema_{period}'] = df['close'].ewm(span=period, adjust=False).mean()
        return df
    
    def check_ema_bounce_signal(self, df: pd.DataFrame, ema_period: int) -> bool:
        if df.empty or f'ema_{ema_period}' not in df.columns:
            return False
        latest = df.iloc[-1]
        ema_value = latest[f'ema_{ema_period}']
        return latest['low'] <= ema_value and latest['close'] > ema_value
    
    def scan_single_stock(self, symbol: str) -> dict:
        df = self.get_stock_bars(symbol)
        if df.empty:
            return None
        
        df = self.calculate_ema(df, 100)
        df = self.calculate_ema(df, 200)
        
        signal_100 = self.check_ema_bounce_signal(df, 100)
        signal_200 = self.check_ema_bounce_signal(df, 200)
        
        if signal_100 or signal_200:
            return {'ticker': symbol, '100ema': signal_100, '200ema': signal_200}
        return None
    
    def scan_multiple_stocks(self, symbols: list) -> pd.DataFrame:
        results = []
        total = len(symbols)
        
        print(f"\n开始扫描 {total} 只股票 (周线级别)...")
        print("-" * 60)
        
        for i, symbol in enumerate(symbols, 1):
            print(f"扫描中 [{i}/{total}]: {symbol}", end="\r")
            signal = self.scan_single_stock(symbol)
            if signal:
                results.append(signal)
        
        print(f"\n扫描完成！共发现 {len(results)} 个信号")
        
        if results:
            return pd.DataFrame(results)
        return pd.DataFrame(columns=['ticker', '100ema', '200ema'])


def format_signals(df: pd.DataFrame) -> pd.DataFrame:
    """格式化信号显示：True显示✅，False不显示"""
    if df.empty:
        return df
    
    df = df.copy()
    df['100ema'] = df['100ema'].apply(lambda x: '✅' if x else '')
    df['200ema'] = df['200ema'].apply(lambda x: '✅' if x else '')
    return df

# ========== 创建保存目录 ==========
save_folder = "ema数据"
if not os.path.exists(save_folder):
    os.makedirs(save_folder)
# ==================================

# 运行周线扫描
scanner = EMAScanner()
weekly_df = scanner.scan_multiple_stocks(watchlist)

today = datetime.now().strftime("%Y-%m-%d")
csv_filename = f"{today}_周级别ema.csv"
csv_path = os.path.join(save_folder, csv_filename)

print("\n📋 周线信号:")
if not weekly_df.empty:
    display_df = format_signals(weekly_df)
    display_df.to_csv(csv_path, index=False)
    print(display_df.to_string(index=False))
else:
    print("  无信号")
    weekly_df.to_csv(csv_path, index=False)

print(f"\n结果已导出: {csv_path}")

In [ ]:
"""搜索
搜索指定 Ticker 是否出现在扫描结果中
"""

import pandas as pd
import os

# ========== 在这里输入要搜索的 Ticker ==========
search_tickers = ["VRT", "ETN","WDC", "STX", "SNDK", "MU","MSFT", "META","NVDA", 
                  "AMD","AVGO", "LITE", "AAOI", "GLW", "CIEN", "COHR","GOOGL", "AMZN"]
# ==============================================

def search_signals(tickers: list):
    tickers = [t.upper() for t in tickers]
    
    print("🔍 搜索 Ticker:", ", ".join(tickers))
    print("=" * 50)
    
    # 检查日线结果
    if os.path.exists("daily_signals.csv"):
        daily_df = pd.read_csv("daily_signals.csv")
        found_daily = daily_df[daily_df['ticker'].isin(tickers)]
        
        print("\n📊 日线结果:")
        if not found_daily.empty:
            for _, row in found_daily.iterrows():
                ema100 = row['100ema'] if pd.notna(row['100ema']) else ""
                ema200 = row['200ema'] if pd.notna(row['200ema']) else ""
                print(f"  {row['ticker']}: 100ema {ema100}  200ema {ema200}")
        else:
            print("  未找到")
    else:
        print("\n📊 日线结果: 没跑 daily")
    
    # 检查周线结果
    if os.path.exists("weekly_signals.csv"):
        weekly_df = pd.read_csv("weekly_signals.csv")
        found_weekly = weekly_df[weekly_df['ticker'].isin(tickers)]
        
        print("\n📊 周线结果:")
        if not found_weekly.empty:
            for _, row in found_weekly.iterrows():
                ema100 = row['100ema'] if pd.notna(row['100ema']) else ""
                ema200 = row['200ema'] if pd.notna(row['200ema']) else ""
                print(f"  {row['ticker']}: 100ema {ema100}  200ema {ema200}")
        else:
            print("  未找到")
    else:
        print("\n📊 周线结果: 没跑 weekly")


search_signals(search_tickers)